In [15]:
import os
import json
import ast
import re
from dotenv import load_dotenv
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor


load_dotenv()
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [16]:
VALID_TARGET_SECTORS = [
    "Aerospazio",
    "Altre società private",
    "Aziende operanti nel settore difesa",
    "Commercio all'ingrosso e al dettaglio",
    "Comunicazione",
    "Energia",
    "Farmaceutico",
    "Fornitura di acqua potabile",
    "Infrastrutture",
    "Intrattenimento",
    "Istruzione",
    "Manifatturiero",
    "Organizzazioni associative",
    "Pubblica amministrazione centrale",
    "Pubblica amministrazione locale",
    "Sanitario",
    "Servizi di sanificazione",
    "Servizi finanziari",
    "Tecnologico",
    "Telecomunicazioni",
    "Trasporti",
    "Università e ricerca",
]

TARGET_SECTORS_PROMPT = "\n".join([f"- {s}" for s in VALID_TARGET_SECTORS])

NIS2_TAXONOMY_GUIDE = """
=== TASSONOMIA UFFICIALE ALLEGATI NIS 2 (D.Lgs. 138/2024) ===

--- ALLEGATO I: Settori ad Alta Criticità (Soggetti Essenziali) ---
- "Energia": Elettricità, sistemi di teleriscaldamento e teleraffreddamento, petrolio, gas, idrogeno.
- "Trasporti": Aereo, ferroviario, marittimo e per vie d'acqua interne, stradale.
- "Settore bancario": Istituti di credito.
- "Infrastrutture dei mercati finanziari": Gestori di sedi di negoziazione e controparti centrali.
- "Sanità": Prestatori di assistenza sanitaria, laboratori di riferimento, ricerca e sviluppo di medicinali, produzione di prodotti farmaceutici di base e specialità farmaceutiche.
- "Acqua potabile": Trattamento, fornitura e distribuzione di acqua potabile.
- "Acque reflue": Raccolta, smaltimento e trattamento delle acque reflue.
- "Infrastrutture digitali": Punti di interscambio internet (IXP), fornitori di servizi DNS, registri di nomi a dominio di primo livello (TLD), fornitori di servizi di cloud computing, fornitori di data center, reti di distribuzione di contenuti (CDN), trust service (prestatori di servizi fiduciari qualificati e non qualificati), comunicazioni elettroniche pubbliche e servizi di accesso a internet.
- "Gestione dei servizi ICT (B2B)": Fornitori di servizi gestiti (MSP) e di sicurezza gestita (MSSP).
- "Pubblica Amministrazione": Amministrazioni pubbliche centrali e regionali.
- "Spazio": Gestori di infrastrutture spaziali a terra.

--- ALLEGATO II: Altri Settori Critici (Soggetti Importanti) ---
- "Servizi postali e di corriere": Servizi postali e corrieri.
- "Gestione dei rifiuti": Raccolta, trattamento e smaltimento dei rifiuti.
- "Fabbricazione, produzione e distribuzione di sostanze chimiche": Chimica di base e specialità chimiche.
- "Produzione, trasformazione e distribuzione di alimenti": Imprese del settore alimentare (produzione, trasformazione e distribuzione all'ingrosso).
- "Fabbricazione": Dispositivi medici e medico-diagnostici in vitro; Computer, prodotti di elettronica e ottica; Apparecchiature elettriche; Macchinari e apparecchiature n.c.a.; Autoveicoli, rimorchi e semirimorchi; Altri mezzi di trasporto (es. nautica, ferroviario, aerospaziale).
- "Fornitori di servizi digitali": Mercati online, motori di ricerca online di dimensioni molto grandi, piattaforme di social network.
- "Organizzazioni di ricerca": Enti e organizzazioni di ricerca.
"""

Funzione detect_triage_scenario

In [17]:
import json
from typing import Any


def _parse_settori_nis_field(raw_field: Any) -> list[dict]:
  """Estrae e normalizza la lista di settori/sottosettori NIS autodichiarati."""
  if not raw_field:
    return []

  if isinstance(raw_field, list):
    if len(raw_field) > 0 and isinstance(raw_field[0], dict):
      return raw_field
    raw_str = ",".join(str(x) for x in raw_field)
  elif isinstance(raw_field, str):
    raw_str = raw_field.strip()
  else:
    return []

  if not raw_str or raw_str.lower() in [
      "nan",
      "none",
      "null",
      "[]",
      "nessun settore",
  ]:
    return []

  # Tentativo parsing JSON
  if raw_str.startswith("[") or raw_str.startswith("{"):
    try:
      parsed = json.loads(raw_str)
      if isinstance(parsed, list):
        return parsed
      if isinstance(parsed, dict):
        return [parsed]
    except Exception:
      pass

  # Parsing da stringhe a virgola o punto e virgola
  risultato = []
  pezzi = [
      p.strip() for p in raw_str.replace(";", ",").split(",") if p.strip()
  ]
  for p in pezzi:
    if "-" in p:
      parti = p.split("-", 1)
      risultato.append(
          {"settore": parti[0].strip(), "sottosettore": parti[1].strip()}
      )
    else:
      risultato.append({"settore": p.strip(), "sottosettore": ""})

  return risultato


def detect_triage_scenario(record_id: str | int) -> dict[str, Any]:
  """Ispeziona deterministicamente il record aziendale e calcola lo scenario operativo di triage."""
  # 1. Recupero del record grezzo dal database
  record = csv_database.get(str(record_id)) or csv_database.get(int(record_id))
  if not record:
    return {
        "scenario": "ERROR_NOT_FOUND",
        "company": {"id": str(record_id), "status": "not_found"},
        "has_target_sector": False,
        "count_nis": 0,
    }

  # 2. Normalizzazione del target_sector
  raw_target = record.get("target_sector")
  if not raw_target or str(raw_target).strip().lower() in [
      "nan",
      "none",
      "null",
      "",
      "nessun business sector trovato",
  ]:
    target_sector = "NON_VALORIZZATO"
    has_target = False
  else:
    target_sector = str(raw_target).strip()
    has_target = True

  # 3. Estrazione e normalizzazione settori NIS
  raw_settori = (
      record.get("settori_nis")
      or record.get("target_sectors")
      or record.get("target_sector_list")
      or []
  )
  settori_nis = _parse_settori_nis_field(raw_settori)
  count_nis = len(settori_nis)

  # 4. Assegnazione logica dello scenario
  if count_nis == 1:
    scenario = "SCENARIO_SINGLE_NIS"
  elif count_nis > 1:
    scenario = "SCENARIO_MULTI_NIS"
  else:
    # 0 settori NIS dichiarati -> Discovery / Non NIS per definizione
    scenario = "SCENARIO_DISCOVERY"

  normalized_company = {
      "id": str(record.get("id", record_id)),
      "denominazione": record.get("denominazione", ""),
      "target_sector": target_sector,
      "settori_nis_autodichiarati": settori_nis,
      "status": "success",
  }

  return {
      "scenario": scenario,
      "company": normalized_company,
      "has_target_sector": has_target,
      "count_nis": count_nis,
  }

In [40]:
import sys
from pathlib import Path

# Risale alla cartella del progetto o posiziona il path relativo corretto
# Se il notebook si trova nella cartella principale dove risiede "sector-lite":
package_path = Path("sector-lite").resolve()

# Aggiunge sector-lite al PYTHONPATH se non già presente
if str(package_path) not in sys.path:
  sys.path.insert(0, str(package_path))

# Ora è possibile importare direttamente dal package sector_lite
from sector_lite.csv_db import CsvDB
from sector_lite.sources import GeminiGroundedSearch

In [71]:
# Inizializzazione delle sorgenti dati sottostanti
filename="anagrafica2.csv"
filepath=""
csv_database = CsvDB("data/anagrafica3.csv",sep=";")
web_searcher = (
    GeminiGroundedSearch() if os.environ.get("GOOGLE_API_KEY") else None
)


# =====================================================================
# TOOL DEFINITIONS
# =====================================================================

def _parse_settori_nis_string(text: str) -> list:
  """Ripulisce stringhe JSON con escape complessi o caratteri di chiusura residui."""
  s = text.strip()
  if not s or s == "[]":
    return []

  # Rimuove apici esterni residui
  while (s.startswith('"') and s.endswith('"')) or (
      s.startswith("'") and s.endswith("'")
  ):
    s = s[1:-1].strip()

  # Rimuove virgolette spaiate dopo la parentesi quadra finale (es. ]")
  if s.endswith(']"') or s.endswith("]'"):
    s = s[:-1]

  # Normalizza backslash ed escape malformati (es. {\settore\": -> {"settore":)
  s = s.replace(r"\"", '"').replace(r"\'", "'")
  s = re.sub(r'\\+([a-zA-Z0-9_]+)"', r'"\1"', s)
  s = re.sub(r"\\+", "", s)

  # Tentativi di deserializzazione
  for parser in (
      lambda x: json.loads(x),
      lambda x: ast.literal_eval(x),
      lambda x: json.loads(x.replace("'", '"')),
  ):
    try:
      res = parser(s)
      if isinstance(res, (list, dict)):
        return res if isinstance(res, list) else [res]
    except Exception:
      continue

  return [{"raw_value": text}]


def get_company_with_scenario(record_id: str | int) -> dict:
  """Recupera l'anagrafica da CSV e determina deterministicamente lo scenario operativo di triage."""
  ctx = detect_triage_scenario(record_id)
  if ctx.get("scenario") in ["ERROR_NOT_FOUND", "NOT_FOUND"]:
    return {
        "status": "not_found",
        "message": f"Nessun record trovato per ID {record_id}",
    }

  company = ctx["company"]
  scenario = ctx["scenario"]
  has_target = ctx["has_target_sector"]
  n_nis = ctx.get("count_nis", ctx.get("num_settori_nis", 0))

  # Direttive operative per gli agenti a valle
  if scenario == "SCENARIO_SINGLE_NIS":
    direttiva = (
        "Sanity check: verificare corrispondenza operativa dell'unico settore"
        " NIS autodichiarato."
    )
  elif scenario == "SCENARIO_MULTI_NIS":
    direttiva = (
        "Disambiguazione: isolare il Core Business principale e scartare"
        " attività secondarie o strumentali interne."
    )
  else:  # SCENARIO_DISCOVERY
    direttiva = (
        "Discovery: l'ente NON ha dichiarato settori NIS ed è FUORI PERIMETRO"
        " NIS 2 per definizione. Ricostruire operatività reale e selezionare il"
        " settore di business dai 22 ammessi (PRIMA_ATTRIBUZIONE)."
    )

  return {
    "status": "success",
    "id": str(company.get("id", record_id)),
    "denominazione": company.get("denominazione", ""),
    "target_sector": company.get("target_sector", "NON_VALORIZZATO"),
    "settori_nis_autodichiarati": company.get(
        "settori_nis_autodichiarati", []
    ),
    "scenario_rilevato": scenario,
    "has_target_sector": has_target,
    "direttiva_operativa": direttiva,
}

def google_search_company(company_name: str, query_focus: str) -> dict:
  """Esegue una ricerca web mirata per scoprire servizi operativi e accreditamenti di un'azienda.

  Args:
      company_name: Denominazione dell'azienda.
      query_focus: Argomento specifico della ricerca (es. 'servizi offerti',
        'accreditamento AgID QTSP').

  Returns:
      Dizionario con lista di risultati, snippet ed evidenze fattuali.
  """
  if not web_searcher:
    return {"status": "error", "message": "Ricerca web non disponibile."}
  query = f"{company_name} {query_focus}"
  results = web_searcher.search(query)
  items = [
      {
          "title": getattr(r, "title", "Web"),
          "url": getattr(r, "url", ""),
          "snippet": getattr(r, "text", "")[:350],
      }
      for r in results[:4]
  ]
  return {"status": "success", "count": len(items), "results": items}

[CsvDB] data/anagrafica3.csv: 7 righe, 5 colonne
[CsvDB] campi riconosciuti: {'id': 'id', 'denominazione': 'denominazione', 'target_sector': 'target_sector', 'target_sectors': 'settori_nis'}
[CsvDB] campi non trovati (opzionali, la pipeline funziona lo stesso senza): ['codice_ateco', 'descrizione', 'dimensione']
[CsvDB] colonne del CSV non usate: ['guid_holistic']


Definizione degli agenti:\
db_retriever_agent: cerca nel db\
researcher_agent: ricerca tramite google search\
classifier_agent: classifica il settore di business ed il settore_nis\
verifier_agent: verifica il risultato del classifier_agent\
business_audit_agent: verifica il settore di business\
pipeline_triage= Agente Sequenziale



In [72]:
db_retriever_agent = LlmAgent(
    name="db_retriever_agent",
    model="gemini-2.5-flash",
    instruction="""Sei l'agente di recupero anagrafico e qualificazione iniziale del record.

Ricevi una richiesta con un `record_id`.

COMPITI:
1. Usa `get_company_with_scenario` per estrarre anagrafica e scenario di triage.
2. Emetti una Scheda Anagrafica chiara e strutturata che guidi i successivi agenti:

FORMATO OBBLIGATORIO OUTPUT:
Scheda Anagrafica Aziendale:
- ID: [id]
- Denominazione: [denominazione]
- Settore Business Dichiarato (target_sector): [target_sector]
- Settori/Sottosettori NIS 2 Autodichiarati: [elenco o 'Nessun settore NIS autodichiarato']
- Scenario Triage: [SCENARIO_SINGLE_NIS | SCENARIO_MULTI_NIS | SCENARIO_DISCOVERY]
- Direttiva Operativa: [direttiva_operativa]
""",
    tools=[get_company_with_scenario],
)

In [73]:
researcher_agent = LlmAgent(
    name="researcher_agent",
    model="gemini-2.5-flash",
    instruction=f"""Sei l'analista investigativo incaricato di accertare l'operatività reale dell'ente tramite ricerche web mirate.

TASSONOMIA DEI 22 SETTORI AMMESSI:
{TARGET_SECTORS_PROMPT}

GUIDA TAXONOMY NIS 2 (D.Lgs. 138/2024):
{NIS2_TAXONOMY_GUIDE}

COMPORTAMENTO IN BASE ALLO SCENARIO:

1. **Se lo scenario è SCENARIO_DISCOVERY (zero settori NIS dichiarati):**
   - L'ente è convenzionalmente FUORI PERIMETRO NIS 2 per assenza di qualificazione a sistema.
   - NON valutare né citare gli Allegati I o II di NIS 2.
   - Concentrati al 100% sulla realtà economica: cosa fa, cosa vende, quali servizi eroga a cittadini/imprese.
   - Formato output obbligatorio:
     **0. Scenario identificato:** SCENARIO_DISCOVERY
     **1. Profilo Operativo Reale:** [Descrizione oggettiva dei servizi e prodotti erogati con fonti]
     **2. Settore di Business Assegnato (dai 22 ammessi):** [Uno dei 22 settori con motivazione economica]
     **3. Valutazione NIS 2:** NON APPLICABILE (Soggetto privo di autodichiarazione a sistema).

2. **Se lo scenario è SCENARIO_SINGLE_NIS o SCENARIO_MULTI_NIS:**
   - Esegui la verifica puntuale della corrispondenza operativa rispetto ai settori NIS autodichiarati.
   - Compila tutti i punti (0, 1, 2 e 3 con riferimento specifico agli Allegati I/II).
""",
    tools=[google_search_company],
)
print("Research Agent defined")

Research Agent defined


In [74]:
classifier_agent = LlmAgent(
    name="classifier_agent",
    model="gemini-2.5-flash",
    instruction=f"""Sei l'esperto semantico e di conformità NIS 2.
Ricevi l'anagrafica aziendale e il report di `researcher_agent`, che include lo scenario identificato
(SCENARIO_A_SINGLE_NIS | SCENARIO_B_MULTI_NIS | SCENARIO_C_DISCOVERY) e, nel caso C, una valutazione
sull'Allegato I/II basata sull'attività reale riscontrata via web.

Ricevi la Scheda Anagrafica e il report di `researcher_agent`.

REGOLA MANDATORIA PER SCENARIO_DISCOVERY:
Se l'ente non ha settori NIS autodichiarati (SCENARIO_DISCOVERY), è convenzionalmente e formalmente FUORI PERIMETRO.
Non devi compiere alcuna qualificazione autonoma sugli Allegati I o II.
Restituisci tassativamente:
  "chiave_principale": null,
  "chiavi_secondarie": [],
  "corrispondenza_attivita": "NON_APPLICABILE",
  "descrizione_sintetica": "Ente escluso dal perimetro NIS 2 per assenza di autodichiarazione a sistema (regola convenzionale).",
  "necessita_revisione_umana": false

TASSONOMIA UFFICIALE NIS 2:
{NIS2_TAXONOMY_GUIDE}

PRINCIPIO GENERALE: l'autodichiarazione dei settori nis a sistema è un punto di partenza che diamo per assodato.

COMPITO OPERATIVO, IN BASE ALLO SCENARIO IDENTIFICATO DAL RESEARCHER:

1. **SCENARIO_A_SINGLE_NIS (1 settore autodichiarato):**
   - Verifica la corrispondenza con l'operatività reale riscontrata dal researcher.
   -Se corrisponde corrispondenza_attività: 'CONFERMATA'
   - Se non corrisponde, dillo esplicitamente: non correggerlo tu, segnala il
   disallineamento, segnala corrispondenza_attività: 'NON CONFERMATA'
2. **SCENARIO_B_MULTI_NIS (più settori autodichiarati):**
   - Assegna a `chiave_principale` il core business prevalente a mercato individuato dal researcher.
   - Assegna a `chiavi_secondarie` gli altri settori validi, escludendo le attività meramente
     strumentali interne.
   - `corrispondenza_attivita`: 'CONFERMATA' o 'PARZIALE'.



3. **In ogni scenario**, se assegni una `chiave_principale`, riporta in `descrizione_sintetica` la voce
   esatta della tassonomia NIS 2 usata e la fonte (autodichiarazione, o citazione del researcher).

RISPONDI ESCLUSIVAMENTE CON QUESTO JSON:
{{
  "chiave_principale": null,
  "chiavi_secondarie": [],
  "corrispondenza_attivita": "CONFERMATA | PARZIALE | NON_CONFORME | NON_APPLICABILE | ATTRIBUITA_DA_RICERCA | NECESSITA_VERIFICA",
  "descrizione_sintetica": "Sintesi dell'inquadramento normativo, con la voce di tassonomia e la fonte usate.",
  "settore di business": "Il settore dell'azienda/amministrazione che più si avvicina alla descrizione"
  "necessita_revisione_umana": true
}}

`necessita_revisione_umana` è true quando `corrispondenza_attivita` è 'ATTRIBUITA_DA_RICERCA' o
'NECESSITA_VERIFICA' (una prima attribuzione basata solo su ricerca web merita una verifica umana prima
di essere considerata definitiva), altrimenti false.""",
    tools=[],
)
print("Classifier agent defined")

Classifier agent defined


In [75]:
verifier_agent = LlmAgent(
    name="verifier_agent",
    model="gemini-2.5-flash",
    instruction=f"""Sei il revisore legale indipendente ("Advocatus Diaboli") per la conformità NIS 2
(D.Lgs. 138/2024). Il tuo compito è cercare attivamente motivi per correggere la proposta, non
confermarla per default.

Ricevi: l'anagrafica, il report di `researcher_agent` (con lo scenario e, nel caso C, la voce di
Allegato I/II citata) e la proposta di `classifier_agent`.

REGOLA TASSATIVA - SCENARIO NON_APPLICABILE (Convenzione Formale):
Se `corrispondenza_attivita` è "NON_APPLICABILE" (ovvero l'ente è in Discovery / privo di dichiarazione a sistema):
- HAI IL DIVIETO ASSOLUTO DI CONTESTARE L'ESCLUSIONE O DI ASSEGNARE CHIAVI NIS DAGLI ALLEGATI, anche se l'ente è un ministero o una utility idrica/energetica.
- L'assenza di autodichiarazione costituisce causa formale e vincolante di esclusione dalla presente procedura di audit.
- Devi ratificare l'esclusione emettendo ESCLUSIVAMENTE questo JSON:
  "approvato": true,
  "chiave_principale_validata": null,
  "chiavi_secondarie_validate": [],
  "corrispondenza_attivita_validata": "NON_APPLICABILE",
  "necessita_revisione_umana": false,
  "audit_validato": 
    "confermati": ["Nessuno (Non applicabile)"],
    "non_confermati": [],
    "omessi": []
  ,
  "giudizio_critico": "Confermata formalmente la non applicabilità della procedura NIS 2 per assenza di qualificazione dell'ente a sistema.",
  "confidenza": "alta"

REVISIONE PER GLI ALTRI SCENARI

TASSONOMIA UFFICIALE NIS 2:
{NIS2_TAXONOMY_GUIDE}

COMPITI DI REVISIONE, IN BASE A `corrispondenza_attivita` PROPOSTA DAL CLASSIFIER:

1. **NON_APPLICABILE (Scenario C - Fuori perimetro):**
   - L'ente non presenta settori NIS autodichiarati a sistema: è convenzionalmente FUORI PERIMETRO.
   - Ratifica formalmente l'esclusione impostando tassativamente:
     • "approvato": true
     • "chiave_principale_validata": null
     • "chiavi_secondarie_validate": []
     • "giudizio_critico": "Confermata la non applicabilità del regime NIS 2 per assenza di qualificazione a sistema."
   - Non eseguire ulteriori contestazioni legali.

2. **CONFERMATA / NON_CONFORME (Scenario A - Singolo settore dichiarato):**
   - Rileggi il profilo operativo del researcher: la corrispondenza (o non corrispondenza) dichiarata dal classifier regge davvero, o il researcher aveva riportato incertezze o discrepanze che il classifier ha trascurato?
   - Se confermata: valida la chiave. Se non conforme: contesta motivando nel giudizio critico.

3. **PARZIALE / DISAMBIGUAZIONE (Scenario B - Più settori dichiarati):**
   - Verifica che `chiave_principale` coincida con il core business effettivo prevalente a mercato descritto dal researcher.
   - Accerta tassativamente che in `chiavi_secondarie_validate` non siano state incluse attività meramente captive o strumentali interne (es. IT interno, autoproduzione).

4. **NECESSITA_VERIFICA:**
   - I dati operativi non consentono di sciogliere il dubbio normativo.
   - Non forzare una conclusione in nessuna direzione: conferma lo stato di sospensione, imposta "approvato": false (o richiedi integrazione) ed evidenzia le lacune in "giudizio_critico".

REGOLA SUL FLAG DI REVISIONE UMANA: puoi alzare `necessita_revisione_umana` da false a true se hai dubbi
residui, ma non abbassarlo da true a false a meno che tu non abbia trovato una conferma autodichiarata
esplicita (non solo la tua rilettura del report web).

RISPONDI ESCLUSIVAMENTE CON QUESTO JSON:
{{
  "approvato": true,
  "chiave_principale_validata": "settore_convalidato" | null,
  "chiavi_secondarie_validate": [],
  "corrispondenza_attivita_validata": "CONFERMATA | PARZIALE | NON_CONFORME | NON_APPLICABILE | ATTRIBUITA_DA_RICERCA | NECESSITA_VERIFICA",
  "necessita_revisione_umana": true,
  "audit_validato": {{
    "confermati": ["elenco settori confermati o 'Nessuno (Non applicabile)'"],
    "non_confermati": [],
    "omessi": []
  }},
  "giudizio_critico": "Sintesi analitica del giudizio di revisione, incluso l'esito del controllo sulla citazione se scenario C.",
  "confidenza": "alta | media | bassa"
}}""",
    tools=[],
)

Agente normalizzatore del business sector con generazione del settore suggerito, come è stato attribuito, la motivazione

In [76]:
business_classifier_agent = LlmAgent(
    name="business_classifier_agent",
    model="gemini-2.5-flash",
    instruction=f"""Sei l'analista merceologico incaricato di determinare il settore economico primario dell'azienda.
Ricevi l'anagrafica, il report operativo di `researcher_agent` e l'esito della validazione NIS 2 di `verifier_agent`.

TASSONOMIA DEI 22 SETTORI AMMESSI:
{TARGET_SECTORS_PROMPT}

METODOLOGIA DI CLASSIFICAZIONE:
1. **Verifica Esito NIS 2 (`verifier_agent`):**
   - **Caso Soggetto NIS Validato (`chiave_principale_validata` presente):**
     Il verifier ha già accertato il Core Business prevalente dell'ente. Usa questo dato come fonte autorevole e mappalo sul settore corrispondente tra i 22 ammessi (es. Sanità -> 'Sanitario', Reti di trasporto -> 'Trasporti', Rifiuti -> 'Gestione dei rifiuti').
   - **Caso Soggetto NON NIS (`chiave_principale_validata: null` o Fuori Perimetro):**
     Determina il settore in piena autonomia basandoti unicamente sulle attività reali e sui servizi prevalenti descritti dal `researcher_agent`.

2. **Regole di Disambiguazione Merceologica:**
   - Gestore vs Fornitore/Manutentore: Chi fa manutenzione o installazione per terzi non appartiene alla filiera servita (officine/gommisti -> 'Altre società private', MAI 'Trasporti'; installatori elettrici/fotovoltaici -> 'Fabbricazione' o 'Altre società private', MAI 'Energia').
   - Servizi IT e Software -> 'Servizi digitali'. Consulenze amministrative/legali -> 'Altre società private'.
   - Commercio all'ingrosso di macchinari industriali -> 'Commercio all'ingrosso'.

3. **Indipendenza dal DB Legacy:**
   NON lasciarti condizionare dal `target_sector` storico presente nella scheda anagrafica: il tuo verdetto deve scaturire dalla coerenza tra il settore validato dal verifier e le evidenze operative del researcher.

RISPONDI ESCLUSIVAMENTE CON QUESTO JSON:
{{
  "settore_business_individuato": "ESATTAMENTE UNO DEI 22 SETTORI AMMESSI",
  "fonte_decisionale": "SETTORE_NIS_VALIDATO | OPERATIVITA_REALE_RESEARCHER",
  "ragione_economica": "Spiega sinteticamente il raccordo tra l'attività reale/NIS accertata e il settore dei 22 scelto."
}}
""",
    tools=[],
)

In [77]:
business_sector_auditor_agent = LlmAgent(
    name="business_sector_auditor_agent",
    model="gemini-2.5-flash",
    instruction=f"""Sei il revisore finale dell'anagrafica merceologica aziendale.
Ricevi il target_sector originario (può essere 'NON_VALORIZZATO'), il profilo del researcher e l'esito
di classifier_agent/verifier_agent (chiave_principale_validata, settore NIS 2).

TASSONOMIA OBBLIGATORIA (22 SETTORI AMMESSI):
{TARGET_SECTORS_PROMPT}

REGOLE DI RISPOSTA:

1. **Se il valore originario è 'NON_VALORIZZATO':**
   - Se il researcher ha trovato evidenza sufficiente per assegnare un settore (ricerca web eseguita,
     con risultati concreti): `esito_valutazione` = 'PRIMA_ATTRIBUZIONE', `settore_business_suggerito`
     = il settore individuato dal researcher, tra i 22 ammessi. Se l'attività è generica o locale senza
     una corrispondenza specifica migliore tra i 22, usa 'Altre società private' come categoria di
     default — questo è un principio generale, non un'eccezione per un solo tipo di attività.
   - Se il researcher segnala che la ricerca web non era disponibile o non ha dato risultati utili:
     `esito_valutazione` = 'NON_DETERMINABILE', `settore_business_suggerito` = null. Non indovinare un
     settore plausibile solo per compilare il campo.

2. **Se il valore originario era già popolato:**
   - `esito_valutazione`: 'COERENTE' (il valore regge rispetto al profilo reale), 'TROPPO_GENERICO'
     (corretto ma impreciso rispetto ai 22 ammessi) o 'ERRATO' (contraddetto dal profilo reale).
   - `settore_business_suggerito`: il settore corretto dai 22 ammessi (uguale all'originale se COERENTE).

3. **In entrambi i casi**, se `chiave_principale_validata` del verifier è valorizzata, verifica che il
   settore di business che stai per suggerire sia ragionevolmente coerente con quel settore NIS 2 (es.
   un ente in Allegato I "Sanità" ci si aspetta in business sector 'Sanitario', non 'Tecnologico'). Se
   non lo è, segnalalo in `note_incoerenza` invece di correggere silenziosamente uno dei due giudizi:
   sono due assi di classificazione diversi, un disallineamento va segnalato a chi legge, non risolto
   da te in autonomia.

4. **Cita sempre la base della tua conclusione** in `fonte`: 'autodichiarazione_originale' se confermi
   il valore a DB, 'profilo_researcher' se ti basi sulla ricerca web, oppure 'evidenza_insufficiente' nel
   caso NON_DETERMINABILE.

RISPONDI ESCLUSIVAMENTE CON UN JSON:
{{
  "target_sector_originale": "valore a DB",
  "esito_valutazione": "COERENTE | TROPPO_GENERICO | ERRATO | PRIMA_ATTRIBUZIONE | NON_DETERMINABILE",
  "settore_business_suggerito": "UNO DEI 22 SETTORI AMMESSI" | null,
  "fonte": "autodichiarazione_originale | profilo_researcher | evidenza_insufficiente",
  "motivazione": "Spiegazione sintetica",
  "note_incoerenza": "Eventuale disallineamento con la classificazione NIS 2, o null",
  "necessita_revisione_umana": true
}}

`necessita_revisione_umana` è true quando `esito_valutazione` è 'PRIMA_ATTRIBUZIONE',
'NON_DETERMINABILE', o 'ERRATO', oppure quando `note_incoerenza` non è null.""",
    tools=[],
)

business_auditor_agent = LlmAgent(
    name="business_auditor_agent",
    model="gemini-2.5-flash",
    instruction=f"""Sei l'Auditor Responsabile della Qualità dei Dati Anagrafici.
Il tuo obiettivo è bonificare il database dai frequenti errori storici di attribuzione merceologica.

Ricevi:
- Il `target_sector` originale registrato a database nella Scheda Anagrafica (può essere 'NON_VALORIZZATO', vuoto, nullo o un settore pregresso).
- Il `settore_business_individuato` proposto dal `business_classifier_agent`.

CRITERI DI CONFRONTO E VERDETTO:
1. **PRIMA_ATTRIBUZIONE**: Se a database il dato era assente, nullo o 'NON_VALORIZZATO'.
2. **COERENTE**: Se il settore a database coincide semanticamente o esattamente con il settore individuato dal classifier indipendente.
3. **TROPPO_GENERICO**: Se a database è presente una voce generica (es. 'Altre società private' o 'Servizi') mentre l'attività economica reale corrisponde a una categoria industriale specifica dei 22 (es. 'Fabbricazione', 'Chimica', 'Gestione dei rifiuti').
4. **ERRATO**: Se il settore a database indica una filiera o settore completamente scorretto (es. una ditta di noleggio/officina classificata storicamente come 'Trasporti', o un'azienda farmaceutica indicata come 'Energia').

VINCOLO:
`settore_business_suggerito` DEVE appartenere alla lista ufficiale dei 22 settori ammessi.

RISPONDI ESCLUSIVAMENTE CON QUESTO JSON:
{{
  "target_sector_originale": "valore a DB",
  "esito_valutazione": "COERENTE | TROPPO_GENERICO | ERRATO | PRIMA_ATTRIBUZIONE",
  "motivazione_audit": "Dettaglio del motivo per cui il dato storico è confermato, contestato o attribuito per la prima volta.",
  "settore_business_suggerito": "UNO DEI 22 SETTORI AMMESSI"
}}
""",
    tools=[],
)

Generazione della pipeline come sequential agent

In [78]:
from google.adk.agents import SequentialAgent
pipeline = SequentialAgent(
    name="pipeline",
    sub_agents=[
        db_retriever_agent,
        researcher_agent,
        classifier_agent,
        verifier_agent,
        business_classifier_agent,
        business_auditor_agent,
    ],
)

C:\Users\Daniele\AppData\Local\Temp\ipykernel_30600\2133304302.py:2: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  pipeline = SequentialAgent(


In [95]:
def extract_last_json(text: str) -> dict:
  matches = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
  if matches:
    try:
      return json.loads(matches[-1])
    except Exception:
      pass
  start, end = text.find("{"), text.rfind("}")
  if start != -1 and end != -1 and end > start:
    try:
      return json.loads(text[start : end + 1])
    except Exception:
      pass
  return {}

In [96]:
runner = InMemoryRunner(agent=pipeline)
#input_data = prepare_triage_input(record_id)
risultati = []

for record_id in range(7, 9):
  print(f"\n========== AVVIO RECORD {record_id} ==========")
  prompt = f"Avvia il triage per record_id: '{record_id}'"

  events = await runner.run_debug(prompt)

  verifier_data = {}
  b_classifier_data = {}
  b_auditor_data = {}

  for ev in events:
    author = getattr(ev, "author", "") or ""
    content = str(getattr(ev, "content", "") or "")

    if "verifier_agent" in author:
      verifier_data = extract_last_json(content)
    elif "business_classifier_agent" in author:
      b_classifier_data = extract_last_json(content)
    elif "business_auditor_agent" in author:
      b_auditor_data = extract_last_json(content)

  risultati.append({
      "id": str(record_id),
      "nis_chiave": verifier_data.get("chiave_principale_validata"),
      "nis_esito": verifier_data.get(
          "corrispondenza_attivita_validata", "NON_APPLICABILE"
      ),
      "settore_proposto": b_classifier_data.get("settore_business_individuato"),
      "target_db_originale": b_auditor_data.get("target_sector_originale"),
      "esito_audit": b_auditor_data.get("esito_valutazione"),
      "settore_finale": b_auditor_data.get("settore_business_suggerito"),
  })

# Tabella riassuntiva a video
df_batch = pd.DataFrame(risultati)
df_batch


========== AVVIO RECORD 7 ==========
db_retriever_agent > Scheda Anagrafica Aziendale:
- ID: 7
- Denominazione: Ministero degli Affari Esteri e della Cooperazione Internazionale
- Settore Business Dichiarato (target_sector): Pubblica amministrazione centrale
- Settori/Sottosettori NIS 2 Autodichiarati: Nessun settore NIS autodichiarato
- Scenario Triage: SCENARIO_DISCOVERY
- Direttiva Operativa: Discovery: l'ente NON ha dichiarato settori NIS ed è FUORI PERIMETRO NIS 2 per definizione. Ricostruire operatività reale e selezionare il settore di business dai 22 ammessi (PRIMA_ATTRIBUZIONE).
researcher_agent > **0. Scenario identificato:** SCENARIO_DISCOVERY
**1. Profilo Operativo Reale:** Il Ministero degli Affari Esteri e della Cooperazione Internazionale (MAECI) è il dicastero del governo italiano incaricato di attuare la politica estera dell'Italia e di rappresentare il Paese nelle relazioni internazionali. La sua missione include la gestione dei rapporti politici, economici, sociali 

,id,nis_chiave,nis_esito,settore_proposto,target_db_originale,esito_audit,settore_finale
0,7,None,NON_APPLICABILE,Pubblica amministrazione centrale,Pubblica amministrazione centrale,COERENTE,Pubblica amministrazione centrale
1,8,None,NON_APPLICABILE,Fornitura di acqua potabile,Fornitura di acqua potabile,COERENTE,Fornitura di acqua potabile
